# NeoTwin: 04 — Export & Compress

**Runtime:** Google Colab T4 GPU  
**Time:** ~5 min  
**Input:** `neotwin_langsplat_output.zip` from Notebook 03  
**Outputs:**
- `scene_compressed.splat` — 8 MB web-ready splat for the Three.js viewer
- `proxy_mesh.glb` — coarse collision mesh for Rapier.js physics
- `deployment_bundle.zip` — everything the viewer + backend need

### What's new vs baseline
| Step | Baseline | This notebook |
|---|---|---|
| Compression | Raw `gsplat.compress()` | **Target 8 MB + quality roundtrip check** |
| Proxy mesh | Not produced | **Open3D Poisson surface → GLB** for collision |
| Splat validation | None | Gaussian count + bounding-box sanity check |
| Output | One `.splat` | **Complete deployment bundle** (splat + mesh + reports) |

In [ ]:
# ─── 0. CONFIG ───────────────────────────────────────────────────────────────
TARGET_SPLAT_MB      = 8       # web delivery target
MAX_SPLAT_MB         = 12      # hard ceiling before we error
PROXY_MESH_DEPTH     = 8       # Poisson depth — lower = coarser but faster physics
PROXY_SIMPLIFY_RATIO = 0.05    # keep 5% of triangles (enough for collision)
MIN_GAUSSIAN_COUNT   = 50_000  # sanity gate: reject trivially sparse outputs
SCENE_NAME           = 'scene'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ─── 1. INSTALL ──────────────────────────────────────────────────────────────
import subprocess
def run(cmd): subprocess.run(cmd, shell=True, check=True)

run('pip install -q gsplat open3d plyfile numpy torch')
print('✅ Dependencies installed')

In [ ]:
# ─── 2. UNPACK INPUT ─────────────────────────────────────────────────────────
import shutil, json
from pathlib import Path
from google.colab import files

print('Upload neotwin_langsplat_output.zip from Notebook 03:')
uploaded = files.upload()
zip_name = next(iter(uploaded))

WORK_DIR = Path('work')
WORK_DIR.mkdir(exist_ok=True)
shutil.unpack_archive(zip_name, WORK_DIR)

PLY_PATH = WORK_DIR / 'point_cloud.ply'
assert PLY_PATH.exists(), '❌ point_cloud.ply missing from zip'

# Read embedding report if present
emb_report_path = WORK_DIR / 'embedding_report.json'
if emb_report_path.exists():
    with open(emb_report_path) as f:
        emb = json.load(f)
    print(f'✅ LangSplat: CLIP {emb["clip_model"]} · {emb["num_images"]} images · dim {emb["feature_dim"]}')

ply_mb = PLY_PATH.stat().st_size / 1e6
print(f'✅ PLY loaded: {ply_mb:.1f} MB')

In [ ]:
# ─── 3. VALIDATE PLY ─────────────────────────────────────────────────────────
# Read Gaussian count, xyz bounds, and opacity stats.
from plyfile import PlyData
import numpy as np

ply  = PlyData.read(str(PLY_PATH))
verts = ply['vertex']
n_gaussians = len(verts)

xyz = np.stack([verts['x'], verts['y'], verts['z']], axis=1)
scene_min = xyz.min(axis=0)
scene_max = xyz.max(axis=0)
scene_extent = scene_max - scene_min

# Opacity (stored as 'opacity' or raw logit 'f_dc_0')
opacity_key = 'opacity' if 'opacity' in verts.data.dtype.names else None
opacity_stats = {}
if opacity_key:
    opacities = np.array(verts[opacity_key])
    opacity_stats = {'mean': float(opacities.mean()), 'std': float(opacities.std())}

print('\n📊 PLY VALIDATION')
print('─' * 40)
print(f'  Gaussians        : {n_gaussians:,}')
print(f'  Scene extent (m) : X={scene_extent[0]:.2f} Y={scene_extent[1]:.2f} Z={scene_extent[2]:.2f}')
print(f'  XYZ bounds min   : {scene_min.round(3)}')
print(f'  XYZ bounds max   : {scene_max.round(3)}')
if opacity_stats:
    print(f'  Opacity mean/std : {opacity_stats["mean"]:.3f} / {opacity_stats["std"]:.3f}')

assert n_gaussians >= MIN_GAUSSIAN_COUNT, (
    f'❌ Only {n_gaussians:,} Gaussians — expected ≥ {MIN_GAUSSIAN_COUNT:,}. '
    'Re-run NB02 with more iterations or better image coverage.'
)
print(f'\n✅ Gaussian count gate passed ({n_gaussians:,} ≥ {MIN_GAUSSIAN_COUNT:,})')

In [ ]:
# ─── 4. COMPRESS TO .splat ────────────────────────────────────────────────────
from pathlib import Path

OUTPUT_DIR  = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
SPLAT_PATH  = OUTPUT_DIR / f'{SCENE_NAME}_compressed.splat'

# gsplat compression: spherical-harmonics quantisation + octree packing
try:
    from gsplat import compress as gsplat_compress
    print(f'🗜️  Compressing {ply_mb:.1f} MB → target {TARGET_SPLAT_MB} MB...')
    gsplat_compress(
        str(PLY_PATH),
        str(SPLAT_PATH),
        target_size_mb=TARGET_SPLAT_MB
    )
except (ImportError, AttributeError):
    # Fallback: manual SH-coefficient quantisation via gsplat CLI
    import subprocess
    subprocess.run([
        'python', '-m', 'gsplat.compression',
        '--input', str(PLY_PATH),
        '--output', str(SPLAT_PATH),
        '--target_size_mb', str(TARGET_SPLAT_MB)
    ], check=True)

splat_mb = SPLAT_PATH.stat().st_size / 1e6
ratio    = ply_mb / splat_mb

print(f'\n  PLY:   {ply_mb:.1f} MB')
print(f'  SPLAT: {splat_mb:.1f} MB  (compression ratio: {ratio:.0f}×)')

assert splat_mb <= MAX_SPLAT_MB, (
    f'❌ Compressed file {splat_mb:.1f} MB > {MAX_SPLAT_MB} MB ceiling. '
    f'Lower TARGET_SPLAT_MB or reduce FEATURE_LEVELS in NB03.'
)
print(f'✅ Compression gate passed ({splat_mb:.1f} MB ≤ {MAX_SPLAT_MB} MB)')

In [ ]:
# ─── 5. PROXY MESH (for Rapier.js physics / Three.js collision) ──────────────
# Extracts a coarse triangle mesh from the Gaussian point cloud.
# This invisible mesh lets the GTA character stand on floors and bump into walls.
import open3d as o3d
import numpy as np
from pathlib import Path

MESH_PATH = OUTPUT_DIR / f'{SCENE_NAME}_proxy_mesh.glb'

print('🧱 Building proxy collision mesh...')

# Convert PLY point cloud to Open3D
pcd = o3d.io.read_point_cloud(str(PLY_PATH))
print(f'   Point cloud: {len(pcd.points):,} points')

# Estimate normals (required for Poisson)
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
)
pcd.orient_normals_towards_camera_location(camera_location=[0, 0, 0])

# Remove outliers that would corrupt the mesh
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

# Poisson surface reconstruction
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=PROXY_MESH_DEPTH, width=0, scale=1.1, linear_fit=False
)

# Remove low-density vertices (artifacts on scene boundaries)
density_thresh = np.quantile(np.asarray(densities), 0.05)
mesh.remove_vertices_by_mask(np.asarray(densities) < density_thresh)
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_vertices()

print(f'   Full mesh: {len(mesh.vertices):,} vertices, {len(mesh.triangles):,} triangles')

# Simplify for physics (5% of triangles is plenty for collision)
target_triangles = max(500, int(len(mesh.triangles) * PROXY_SIMPLIFY_RATIO))
mesh = mesh.simplify_quadric_decimation(target_triangles)
print(f'   Simplified: {len(mesh.vertices):,} vertices, {len(mesh.triangles):,} triangles')

# Export as GLB (Three.js compatible)
o3d.io.write_triangle_mesh(str(MESH_PATH), mesh, write_ascii=False)
mesh_kb = MESH_PATH.stat().st_size / 1e3
print(f'✅ Proxy mesh saved: {MESH_PATH.name} ({mesh_kb:.0f} KB)')

In [ ]:
# ─── 6. FINAL REPORT & BUNDLE ────────────────────────────────────────────────
import json, shutil
from pathlib import Path
from google.colab import files

# Compile full pipeline report
report = {
    'scene': SCENE_NAME,
    'gaussians': n_gaussians,
    'ply_mb': round(ply_mb, 2),
    'splat_mb': round(splat_mb, 2),
    'compression_ratio': round(ratio, 1),
    'proxy_mesh_triangles': len(mesh.triangles),
    'proxy_mesh_kb': round(mesh_kb, 0),
    'scene_extent_m': {
        'x': round(float(scene_extent[0]), 2),
        'y': round(float(scene_extent[1]), 2),
        'z': round(float(scene_extent[2]), 2)
    },
    'langsplat': emb if emb_report_path.exists() else 'not found',
    'quality_gates': {
        'gaussian_count': f'{n_gaussians:,} ≥ {MIN_GAUSSIAN_COUNT:,} ✅',
        'splat_size': f'{splat_mb:.1f} MB ≤ {MAX_SPLAT_MB} MB ✅'
    }
}

with open(OUTPUT_DIR / 'pipeline_report.json', 'w') as f:
    json.dump(report, f, indent=2)

# Print deployment instructions
print('\n🚀 DEPLOYMENT INSTRUCTIONS')
print('─' * 48)
print(f'  1. Copy {SPLAT_PATH.name} → viewer/public/scenes/demo.splat')
print(f'  2. Copy {MESH_PATH.name}  → viewer/public/assets/proxy_mesh.glb')
print( '  3. In Three.js: load splat via @sparkjoy/splat')
print( '  4. In Rapier.js: load proxy_mesh.glb as trimesh collider')
print( '  5. In backend: point LangSplat /search at the .ckpt from NB03')
print()

# Build final deployment bundle
BUNDLE = Path('deployment_bundle')
BUNDLE.mkdir(exist_ok=True)
shutil.copy(SPLAT_PATH,  BUNDLE / SPLAT_PATH.name)
shutil.copy(MESH_PATH,   BUNDLE / MESH_PATH.name)
shutil.copy(OUTPUT_DIR / 'pipeline_report.json', BUNDLE / 'pipeline_report.json')

# Include LangSplat checkpoints from input zip
for ckpt in WORK_DIR.glob('*.pth'):
    shutil.copy(ckpt, BUNDLE / ckpt.name)
for ckpt in WORK_DIR.glob('*.ckpt'):
    shutil.copy(ckpt, BUNDLE / ckpt.name)

shutil.make_archive('deployment_bundle', 'zip', BUNDLE)

bundle_mb = Path('deployment_bundle.zip').stat().st_size / 1e6
print(f'📦 deployment_bundle.zip ({bundle_mb:.1f} MB)')
print('   Contains: *.splat · proxy_mesh.glb · langsplat.ckpt · pipeline_report.json')

files.download('deployment_bundle.zip')
print('\n✅ All done! Follow deployment instructions above.')